# Crawling Data X (>2000 tweet) — Kebijakan WFH ASN Hari Jumat

Notebook ini memakai **tweet-harvest** (oleh Helmi Satria) untuk mengambil data **historis** dari X **tanpa API developer**. Tweet-harvest menjalankan browser Chromium dan login memakai **`auth_token`** akun X kamu.

**Cocok untuk:** mengumpulkan ribuan tweet untuk analisis sentimen, dengan rentang tanggal bebas.

---
### ⚠️ Baca dulu (penting)
- Metode ini **melanggar ToS X** dan ada **risiko akun di-suspend**. **Gunakan akun buangan/sekunder**, jangan akun utama.
- **Jaga kerahasiaan `auth_token`** — jangan dibagikan ke siapa pun. Token ini setara akses ke akun kamu.
- Kalau kena pembatasan, jeda ~30 menit lalu coba lagi.

## 1. Cara mendapatkan `auth_token`

1. Login ke **https://x.com** lewat browser (pakai akun buangan).
2. Tekan **F12** (Developer Tools) → tab **Application** (Chrome) / **Storage** (Firefox).
3. Buka **Cookies → https://x.com**.
4. Cari baris bernama **`auth_token`**, lalu salin **Value**-nya (deretan huruf & angka).

Nilai itulah yang akan kamu tempel pada sel di bawah.

## 2. Install Node.js (tweet-harvest dibangun dengan Node.js)

In [ ]:
!pip install -q pandas

# Pasang Node.js v20
!sudo apt-get update -qq
!sudo apt-get install -y -qq ca-certificates curl gnupg
!sudo mkdir -p /etc/apt/keyrings
!curl -fsSL https://deb.nodesource.com/gpgkey/nodesource-repo.gpg.key | sudo gpg --dearmor -o /etc/apt/keyrings/nodesource.gpg
!echo "deb [signed-by=/etc/apt/keyrings/nodesource.gpg] https://deb.nodesource.com/node_20.x nodistro main" | sudo tee /etc/apt/sources.list.d/nodesource.list
!sudo apt-get update -qq
!sudo apt-get install -y -qq nodejs
!node -v

## 3. Masukkan auth_token (input tersembunyi, tidak tersimpan di notebook)

In [ ]:
from getpass import getpass

twitter_auth_token = getpass('Tempel auth_token kamu lalu tekan Enter: ')
print('Token diterima.' if twitter_auth_token else 'Token kosong!')

## 4. Konfigurasi pencarian & jalankan crawling

- `limit` = target jumlah tweet (set 2000).
- `since`/`until` = rentang tanggal historis (ubah sesuai kebutuhan).
- Hasil otomatis tersimpan di folder `tweets-data/`.

In [ ]:
filename = 'wfh_asn.csv'
limit = 2000

# Kata kunci + filter. Operator sama seperti Pencarian Lanjutan di web X.
search_keyword = (
    '("WFH ASN" OR "WFH PNS" OR "ASN WFH" OR "kerja dari rumah ASN") '
    'lang:id since:2024-01-01 until:2025-12-31'
)

print('Kata kunci:', search_keyword)

!npx --yes tweet-harvest@2.6.1 -o "{filename}" -s "{search_keyword}" --tab "LATEST" -l {limit} --token {twitter_auth_token}

## 5. Muat & periksa hasil

In [ ]:
import os, glob
import pandas as pd

# Cek isi folder tweets-data/ untuk tahu apa yang benar-benar dihasilkan crawling.
if os.path.exists('tweets-data'):
    print('Isi folder tweets-data/:', os.listdir('tweets-data'))
else:
    print('Folder tweets-data/ BELUM ada -> berarti Sel 4 (crawling) belum berhasil.')

candidates = glob.glob('tweets-data/*.csv')
print('File CSV ditemukan:', candidates)

if not candidates:
    raise FileNotFoundError(
        'Tidak ada CSV hasil crawling. Penyebab umum:\n'
        '  1. auth_token salah/kadaluarsa -> ambil ulang dari cookie x.com\n'
        '  2. 0 tweet untuk query/tanggal tsb -> longgarkan kata kunci / perlebar tanggal\n'
        '  3. tweet-harvest error -> lihat output Sel 4\n'
        'Perbaiki lalu jalankan ulang Sel 4.'
    )

# Ambil file CSV terbaru (kalau ada lebih dari satu).
path = max(candidates, key=os.path.getmtime)
df = pd.read_csv(path)
print(f'\nMemuat: {path}')
print(f'Jumlah tweet terkumpul: {len(df)}')
print('Kolom:', list(df.columns))
df.head()

## 6. (Opsional) Analisis sentimen IndoBERT

Kolom teks pada output tweet-harvest biasanya bernama `full_text`.

In [ ]:
!pip install -q transformers torch

import re
from transformers import pipeline

TEXT_COL = 'full_text' if 'full_text' in df.columns else df.columns[-1]
LABEL_MAP = {'LABEL_0': 'positif', 'LABEL_1': 'netral', 'LABEL_2': 'negatif'}

def bersihkan(t):
    t = re.sub(r'http\S+|www\.\S+', '', str(t))
    t = re.sub(r'@\w+', '', t)
    t = re.sub(r'#', '', t)
    return re.sub(r'\s+', ' ', t).strip()

df['clean_text'] = df[TEXT_COL].apply(bersihkan)
df = df[df['clean_text'].str.len() > 3].reset_index(drop=True)

clf = pipeline('sentiment-analysis',
               model='mdhugol/indonesia-bert-sentiment-classification',
               truncation=True, max_length=256)

hasil = clf(df['clean_text'].tolist())
df['sentimen'] = [LABEL_MAP.get(r['label'], r['label']) for r in hasil]
df['skor'] = [round(r['score'], 3) for r in hasil]
df.to_csv('wfh_asn_sentimen.csv', index=False, encoding='utf-8')
print(df['sentimen'].value_counts())
df[[TEXT_COL, 'sentimen', 'skor']].head(10)

In [ ]:
import matplotlib.pyplot as plt

ringkasan = df['sentimen'].value_counts().reindex(['positif', 'netral', 'negatif'])
ringkasan.plot(kind='bar', color=['#2ecc71', '#95a5a6', '#e74c3c'])
plt.title('Sentimen Publik: Kebijakan WFH ASN Hari Jumat')
plt.ylabel('Jumlah Tweet')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('sentimen_wfh_asn.png', dpi=120)
plt.show()